In [19]:
import sys
!uv pip install pandas --python {sys.executable}
!uv pip install selenium --python {sys.executable}
!uv pip install BeautifulSoup4
!uv pip install seaborn
!uv pip install lxml
import lxml
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
import seaborn 

from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

import time
!uv pip install webdriver-manager
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service


options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

Audited 1 package in 4ms
Audited 1 package in 3ms
Audited 1 package in 2ms
Audited 1 package in 4ms
Audited 1 package in 3ms
Audited 1 package in 4ms


In [21]:
driver.quit() # restart the driver because it keeps crashing :)

options = Options()
options.add_argument("--headless=new")  # newer headless mode, more stable
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")  # Shiny needs a real viewport to render
options.add_argument("--remote-debugging-port=9222")

driver = webdriver.Chrome(options=options)
driver.get("https://avrpublic.dhhs.utah.gov/imms_dashboard/")
time.sleep(30)

# Check session is alive
print(driver.title)

Utah's Immunization Dashboard


In [22]:
# Click the tab
tab = WebDriverWait(driver, 30).until(
    EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), 'School')]"))
)
tab.click()

time.sleep(20)  # give Shiny time to load the data

soup = BeautifulSoup(driver.page_source, "html.parser")
tables = soup.find_all("table")
print(f"Found {len(tables)} tables")
for t in tables:
    print(t.get("id"), t.get("class"))

Found 2 tables
None ['display', 'dataTable', 'no-footer']
DataTables_Table_0 ['display', 'dataTable', 'no-footer']


In [23]:
table = soup.find("table", {"id": "DataTables_Table_0"})
df = pd.read_html(StringIO(str(table)))[0]
print(df.shape)
print(df.head())

(11, 7)
                                     Facility name              City  \
0                                              NaN               NaN   
1                                River Rock School              Lehi   
2                                Springside School  Saratoga Springs   
3  American Preparatory Academy Accelerated School  West Valley City   
4                     American Preparatory Academy            Draper   

   Adequately immunized %  Exempt %  Conditionally enrolled %  \
0                     NaN       NaN                       NaN   
1                    89.7      10.3                       0.0   
2                    82.9      13.7                       0.9   
3                    94.2       3.5                       0.0   
4                    92.0       7.0                       1.0   

   Extended conditionally enrolled %  Out of compliance %  
0                                NaN                  NaN  
1                                0.0            

In [ ]:
# See pagination info
print(soup.find(class_="dataTables_info"))   # only shows like 10 entries at a time
print(soup.find(class_="dataTables_paginate"))

None
<div class="dataTables_paginate paging_simple_numbers" id="DataTables_Table_0_paginate"><a aria-controls="DataTables_Table_0" aria-disabled="true" class="paginate_button previous disabled" data-dt-idx="previous" id="DataTables_Table_0_previous" role="link" tabindex="-1">Previous</a><span><a aria-controls="DataTables_Table_0" aria-current="page" class="paginate_button current" data-dt-idx="0" role="link" tabindex="0">1</a><a aria-controls="DataTables_Table_0" class="paginate_button" data-dt-idx="1" role="link" tabindex="0">2</a><a aria-controls="DataTables_Table_0" class="paginate_button" data-dt-idx="2" role="link" tabindex="0">3</a><a aria-controls="DataTables_Table_0" class="paginate_button" data-dt-idx="3" role="link" tabindex="0">4</a><a aria-controls="DataTables_Table_0" class="paginate_button" data-dt-idx="4" role="link" tabindex="0">5</a><span class="ellipsis">…</span><a aria-controls="DataTables_Table_0" class="paginate_button" data-dt-idx="63" role="link" tabindex="0">64<

In [ ]:
all_dfs = [df]  # we already have page 1

for page in range(2, 65):  # pages 2-64
    # clicking the next button 
    next_btn = driver.find_element(By.ID, "DataTables_Table_0_next")
    next_btn.click()
    time.sleep(2)  # give Shiny time to update
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    table = soup.find("table", {"id": "DataTables_Table_0"})
    page_df = pd.read_html(StringIO(str(table)))[0]
    all_dfs.append(page_df)
    print(f"Page {page}/64 done — {len(page_df)} rows")

final_df = pd.concat(all_dfs, ignore_index=True)
print(final_df.shape)
final_df.to_csv("school_vaccine_exemptions.csv", index=False)

Page 2/64 done — 11 rows
Page 3/64 done — 11 rows
Page 4/64 done — 11 rows
Page 5/64 done — 11 rows
Page 6/64 done — 11 rows
Page 7/64 done — 11 rows
Page 8/64 done — 11 rows
Page 9/64 done — 11 rows
Page 10/64 done — 11 rows
Page 11/64 done — 11 rows
Page 12/64 done — 11 rows
Page 13/64 done — 11 rows
Page 14/64 done — 11 rows
Page 15/64 done — 11 rows
Page 16/64 done — 11 rows
Page 17/64 done — 11 rows
Page 18/64 done — 11 rows
Page 19/64 done — 11 rows
Page 20/64 done — 11 rows
Page 21/64 done — 11 rows
Page 22/64 done — 11 rows
Page 23/64 done — 11 rows
Page 24/64 done — 11 rows
Page 25/64 done — 11 rows
Page 26/64 done — 11 rows
Page 27/64 done — 11 rows
Page 28/64 done — 11 rows
Page 29/64 done — 11 rows
Page 30/64 done — 11 rows
Page 31/64 done — 11 rows
Page 32/64 done — 11 rows
Page 33/64 done — 11 rows
Page 34/64 done — 11 rows
Page 35/64 done — 11 rows
Page 36/64 done — 11 rows
Page 37/64 done — 11 rows
Page 38/64 done — 11 rows
Page 39/64 done — 11 rows
Page 40/64 done — 11

In [26]:
final_df

,Facility name,City,Adequately immunized %,Exempt %,Conditionally enrolled %,Extended conditionally enrolled %,Out of compliance %
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,River Rock School,Lehi,89.7,10.3,0.0,0.0,0.0
2,Springside School,Saratoga Springs,82.9,13.7,0.9,0.9,1.7
3,American Preparatory Academy Accelerated School,West Valley City,94.2,3.5,0.0,2.3,0.0
4,American Preparatory Academy,Draper,92.0,7.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...
693,NaN,NaN,NaN,NaN,NaN,NaN,NaN
694,Utah Peak Academy,Junction,50.0,45.0,2.5,0.0,2.5
695,Wallace Stegner Academy Kearns,West Valley City,95.9,4.1,0.0,0.0,0.0
696,Haven Bay,West Haven,89.8,8.5,0.0,1.7,0.0
